# GitLab Issues Fetcher - OPTIMIZED VERSION ⚡

## Performance Improvements:
- **Parallel Link Fetching**: Uses 20 threads to fetch issue links simultaneously
- **Bulk Database Inserts**: Batch processing with 1000 rows per batch
- **Optimized Excel Creation**: Streamlined formatting
- **Progress Tracking**: Real-time execution time monitoring

**Expected Speed Improvement**: 5-10x faster than sequential version

## 1. Import Libraries

In [ ]:
import requests
import pandas as pd
import getpass
from typing import List, Dict, Any, Tuple
from datetime import datetime
import psycopg2
from psycopg2.extras import execute_batch
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
import time
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

## 2. Configuration

In [ ]:
# GitLab Configuration
GITLAB_URL = 'https://devcloud.ubs.net'
IKG_PROJECT_PATH = 'ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-insights-cl/commons/staat-ds-insights-home'
SWAT_PROJECT_PATH = 'ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-insights-cl/commons/staat-ds-insights-cl-home'

# Greenplum Configuration
GREENPLUM_HOST = 'greenplum-rdsp.zur.swissbank.com'
GREENPLUM_PORT = 5432
GREENPLUM_DB = 'gprdsp'
GREENPLUM_USER = 'ds_rdsp_dev'
GREENPLUM_SCHEMA = 'sandbox_prj_smart_insights'

# Table names
OUTPUT_TABLE1 = 'ikg_issue_details'
OUTPUT_TABLE2 = 'swat_issue_details'
ARCHIVE_TABLE1 = 'ikg_issue_details_archive'
ARCHIVE_TABLE2 = 'swat_issue_details_archive'

# Performance settings
MAX_WORKERS = 20  # Parallel threads for link fetching
BATCH_SIZE = 1000  # Database insert batch size

print(f"GitLab URL: {GITLAB_URL}")
print(f"\nPerformance Settings:")
print(f"  Parallel threads: {MAX_WORKERS}")
print(f"  DB batch size: {BATCH_SIZE}")

## 3. Start Timer

In [ ]:
start_time = time.time()
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 4. GitLab Authentication

In [ ]:
gitlab_token = getpass.getpass("Enter your GitLab Personal Access Token: ")

headers = {
    'PRIVATE-TOKEN': gitlab_token,
    'Content-Type': 'application/json'
}

print("✓ GitLab authentication configured")

## 5. Optimized Helper Functions

In [ ]:
def get_project_id(gitlab_url: str, project_path: str, headers: dict) -> Tuple[str, str]:
    encoded_path = requests.utils.quote(project_path, safe='')
    url = f"{gitlab_url}/api/v4/projects/{encoded_path}"
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    project_data = response.json()
    return project_data['id'], project_data['name']


def fetch_issue_links(gitlab_url: str, project_id: str, issue_iid: int, headers: dict) -> List[Dict]:
    """Fetch links for a single issue"""
    url = f"{gitlab_url}/api/v4/projects/{project_id}/issues/{issue_iid}/links"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json()
    except:
        return []


def fetch_links_batch(gitlab_url: str, project_id: str, issue_iids: List[int], 
                     headers: dict, max_workers: int = 20) -> Dict[int, List[Dict]]:
    """OPTIMIZED: Fetch links for multiple issues in parallel"""
    links_map = {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_iid = {
            executor.submit(fetch_issue_links, gitlab_url, project_id, iid, headers): iid 
            for iid in issue_iids
        }
        
        completed = 0
        total = len(issue_iids)
        
        for future in as_completed(future_to_iid):
            iid = future_to_iid[future]
            try:
                links_map[iid] = future.result()
            except:
                links_map[iid] = []
            
            completed += 1
            if completed % 50 == 0:
                print(f"    Fetched links: {completed}/{total}")
    
    return links_map


def fetch_all_issues(gitlab_url: str, project_id: str, project_name: str, 
                    headers: dict, max_workers: int = 20) -> List[Dict]:
    """OPTIMIZED: Fetch issues with parallel link fetching"""
    all_issues = []
    page = 1
    per_page = 100
    url = f"{gitlab_url}/api/v4/projects/{project_id}/issues"
    
    print(f"\nFetching issues from {project_name}...")
    
    # Fetch all issues
    while True:
        params = {
            'per_page': per_page,
            'page': page,
            'state': 'all',
            'scope': 'all',
            'with_labels_details': True
        }
        
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        issues = response.json()
        
        if not issues:
            break
        
        all_issues.extend(issues)
        print(f"  Page {page}: {len(issues)} issues (Total: {len(all_issues)})")
        
        if len(issues) < per_page:
            break
        page += 1
    
    print(f"✓ Fetched {len(all_issues)} issues")
    
    # Fetch links in parallel
    if all_issues:
        print(f"  Fetching links in parallel ({max_workers} threads)...")
        issue_iids = [issue['iid'] for issue in all_issues]
        links_map = fetch_links_batch(gitlab_url, project_id, issue_iids, headers, max_workers)
        
        for issue in all_issues:
            issue['_links_data'] = links_map.get(issue['iid'], [])
        
        print(f"  ✓ Links fetched")
    
    return all_issues

print("✓ Optimized helper functions defined")

In [ ]:
def extract_issue_data(issues: List[Dict], project_identifier: str) -> pd.DataFrame:
    """Extract issue data with consolidated links"""
    extracted_data = []
    current_datetime = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for issue in issues:
        assignees = [a.get('name', '') for a in issue.get('assignees', [])]
        assignee_str = ', '.join(assignees) if assignees else None
        
        labels = issue.get('labels', [])
        if labels:
            if isinstance(labels[0], dict):
                label_str = ', '.join([label.get('name', '') for label in labels])
            else:
                label_str = ', '.join(labels)
        else:
            label_str = None
        
        iteration = issue.get('iteration', {}).get('title', '') if issue.get('iteration') else None
        epic = issue.get('epic', {}).get('title', '') if issue.get('epic') else None
        epic_iid = issue.get('epic', {}).get('iid', '') if issue.get('epic') else None
        participants = [p.get('name', '') for p in issue.get('participants', [])]
        participants_str = ', '.join(participants) if participants else None
        milestone = issue.get('milestone', {}).get('title', '') if issue.get('milestone') else None
        time_estimate = issue.get('time_stats', {}).get('time_estimate')
        time_spent = issue.get('time_stats', {}).get('total_time_spent')
        
        task_completion = None
        if issue.get('task_completion_status'):
            completed = issue['task_completion_status'].get('completed_count', 0)
            total = issue['task_completion_status'].get('count', 0)
            task_completion = f"{completed}/{total}"
        
        links_data = issue.get('_links_data', [])
        link_ids = [str(link.get('id', '')) for link in links_data if link.get('id')]
        link_issue_ids = [str(link.get('issue_link_id', '')) for link in links_data if link.get('issue_link_id')]
        link_issue_iids = [str(link.get('iid', '')) for link in links_data if link.get('iid')]
        link_types = [str(link.get('link_type', '')) for link in links_data if link.get('link_type')]
        link_urls = [str(link.get('web_url', '')) for link in links_data if link.get('web_url')]
        link_issue_titles = [str(link.get('title', '')) for link in links_data if link.get('title')]
        linked_project_ids = [str(link.get('project_id', '')) for link in links_data if link.get('project_id')]
        
        issue_data = {
            'project': project_identifier,
            'issue_id': issue.get('id'),
            'issue_iid': issue.get('iid'),
            'title': issue.get('title'),
            'description': issue.get('description'),
            'state': issue.get('state'),
            'web_url': issue.get('web_url', ''),
            'link_id': ', '.join(link_ids) if link_ids else None,
            'link_issue_id': ', '.join(link_issue_ids) if link_issue_ids else None,
            'link_issue_iid': ', '.join(link_issue_iids) if link_issue_iids else None,
            'link_type': ', '.join(link_types) if link_types else None,
            'link_url': ', '.join(link_urls) if link_urls else None,
            'link_issue_title': ', '.join(link_issue_titles) if link_issue_titles else None,
            'linked_project_id': ', '.join(linked_project_ids) if linked_project_ids else None,
            'author': issue.get('author', {}).get('name'),
            'author_username': issue.get('author', {}).get('username'),
            'created_by_id': issue.get('author', {}).get('id'),
            'assignee': assignee_str,
            'assignee_ids': ', '.join([str(a.get('id', '')) for a in issue.get('assignees', [])]),
            'issue_created_date': issue.get('created_at'),
            'created_at': issue.get('created_at'),
            'updated_at': issue.get('updated_at'),
            'closed_at': issue.get('closed_at'),
            'due_date': issue.get('due_date'),
            'start_date': issue.get('start_date'),
            'current_date_time': current_datetime,
            'labels': label_str,
            'milestone': milestone,
            'iteration': iteration,
            'epic': epic,
            'epic_iid': epic_iid,
            'weight': issue.get('weight'),
            'parent_iid': None,
            'has_tasks': issue.get('has_tasks'),
            'task_completion_status': task_completion,
            'participants': participants_str,
            'upvotes': issue.get('upvotes'),
            'downvotes': issue.get('downvotes'),
            'user_notes_count': issue.get('user_notes_count'),
            'merge_requests_count': issue.get('merge_requests_count'),
            'time_estimate_hours': time_estimate / 3600 if time_estimate else None,
            'time_spent_hours': time_spent / 3600 if time_spent else None,
            'confidential': issue.get('confidential'),
            'discussion_locked': issue.get('discussion_locked'),
            'issue_type': issue.get('issue_type'),
            'severity': issue.get('severity'),
            'health_status': issue.get('health_status'),
        }
        extracted_data.append(issue_data)
    
    return pd.DataFrame(extracted_data)

print("✓ Data extraction function defined")

## 6. Fetch IKG Issues (Parallel)

In [ ]:
step_start = time.time()

ikg_project_id, ikg_project_name = get_project_id(GITLAB_URL, IKG_PROJECT_PATH, headers)
print(f"✓ IKG Project: {ikg_project_name} (ID: {ikg_project_id})")

ikg_issues = fetch_all_issues(GITLAB_URL, ikg_project_id, ikg_project_name, headers, MAX_WORKERS)
ikg_df = extract_issue_data(ikg_issues, 'staat-ds-insights-home')

step_duration = time.time() - step_start
print(f"✓ IKG: {len(ikg_df)} issues in {step_duration:.2f}s")

## 7. Fetch SWAT Issues (Parallel)

In [ ]:
step_start = time.time()

swat_project_id, swat_project_name = get_project_id(GITLAB_URL, SWAT_PROJECT_PATH, headers)
print(f"✓ SWAT Project: {swat_project_name} (ID: {swat_project_id})")

swat_issues = fetch_all_issues(GITLAB_URL, swat_project_id, swat_project_name, headers, MAX_WORKERS)
swat_df = extract_issue_data(swat_issues, 'staat-ds-insights-cl-home')

step_duration = time.time() - step_start
print(f"✓ SWAT: {len(swat_df)} issues in {step_duration:.2f}s")

## 8. Create Excel File

In [ ]:
step_start = time.time()

wb = Workbook()
wb.remove(wb.active)

# IKG sheet
ikg_sheet = wb.create_sheet('IKG')
for r_idx, r in enumerate(dataframe_to_rows(ikg_df, index=False, header=True)):
    ikg_sheet.append(r)
    if r_idx == 0:
        for cell in ikg_sheet[1]:
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')

# SWAT sheet
swat_sheet = wb.create_sheet('SWAT')
for r_idx, r in enumerate(dataframe_to_rows(swat_df, index=False, header=True)):
    swat_sheet.append(r)
    if r_idx == 0:
        for cell in swat_sheet[1]:
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')

excel_filename = 'gitlab_issues_ikg_swat.xlsx'
wb.save(excel_filename)

step_duration = time.time() - step_start
print(f"✓ Excel created in {step_duration:.2f}s")

## 9. Greenplum Connection

In [ ]:
db_password = getpass.getpass("Enter Greenplum database password: ")

conn = psycopg2.connect(
    host=GREENPLUM_HOST,
    port=GREENPLUM_PORT,
    database=GREENPLUM_DB,
    user=GREENPLUM_USER,
    password=db_password
)
conn.autocommit = False
print(f"✓ Connected to Greenplum")

## 10. Database Helper Functions

In [ ]:
def create_table(conn, schema: str, table_name: str, df: pd.DataFrame, drop_if_exists: bool = True):
    cursor = conn.cursor()
    try:
        if drop_if_exists:
            cursor.execute(f"DROP TABLE IF EXISTS {schema}.{table_name}")
        
        columns = []
        for col, dtype in df.dtypes.items():
            col_type = 'TEXT' if dtype == 'object' else 'BIGINT' if dtype == 'int64' else 'DOUBLE PRECISION' if dtype == 'float64' else 'BOOLEAN' if dtype == 'bool' else 'TEXT'
            columns.append(f"{col} {col_type}")
        
        cursor.execute(f"CREATE TABLE {schema}.{table_name} ({', '.join(columns)}) DISTRIBUTED RANDOMLY")
        conn.commit()
        print(f"  ✓ Created {schema}.{table_name}")
    finally:
        cursor.close()


def insert_data_bulk(conn, schema: str, table_name: str, df: pd.DataFrame, batch_size: int = 1000):
    """OPTIMIZED: Bulk insert with batching"""
    cursor = conn.cursor()
    try:
        columns_str = ', '.join(df.columns)
        placeholders = ', '.join(['%s'] * len(df.columns))
        insert_query = f"INSERT INTO {schema}.{table_name} ({columns_str}) VALUES ({placeholders})"
        
        data = [tuple(x) for x in df.to_numpy()]
        execute_batch(cursor, insert_query, data, page_size=batch_size)
        conn.commit()
        print(f"  ✓ Inserted {len(df)} rows")
    finally:
        cursor.close()

print("✓ Database functions defined")

## 11. Create and Load Tables

In [ ]:
step_start = time.time()

print("Creating main tables...")
create_table(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE1, ikg_df, True)
create_table(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE2, swat_df, True)

print("\nCreating archive tables...")
create_table(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE1, ikg_df, False)
create_table(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE2, swat_df, False)

print("\nInserting into main tables (bulk)...")
insert_data_bulk(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE1, ikg_df, BATCH_SIZE)
insert_data_bulk(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE2, swat_df, BATCH_SIZE)

print("\nInserting into archive tables (bulk)...")
insert_data_bulk(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE1, ikg_df, BATCH_SIZE)
insert_data_bulk(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE2, swat_df, BATCH_SIZE)

step_duration = time.time() - step_start
print(f"\n✓ Database operations completed in {step_duration:.2f}s")

## 12. Cleanup

In [ ]:
conn.close()
print("✓ Connection closed")

## 13. Final Summary with Timing

In [ ]:
total_duration = time.time() - start_time

print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"\nIKG Issues: {len(ikg_df)} rows")
print(f"SWAT Issues: {len(swat_df)} rows")
print(f"\nExcel: {excel_filename}")
print(f"\nGreenplum Tables:")
print(f"  - {GREENPLUM_SCHEMA}.{OUTPUT_TABLE1}")
print(f"  - {GREENPLUM_SCHEMA}.{OUTPUT_TABLE2}")
print(f"  - {GREENPLUM_SCHEMA}.{ARCHIVE_TABLE1}")
print(f"  - {GREENPLUM_SCHEMA}.{ARCHIVE_TABLE2}")
print(f"\n⏱️  TOTAL TIME: {total_duration:.2f} seconds ({total_duration/60:.2f} minutes)")
print(f"\n✓ Process completed successfully!")
print("=" * 80)